# Instalar dependencias

In [ ]:
!pip install -q ragas langchain langgraph langchain-openai qdrant-client python-dotenv datasets xlsxwriter

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 457.4/457.4 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.7/84.7 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 377.2/377.2 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 489.1/489.1 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.9/160.9 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 358.8/358.8 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 25.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 226.4/2

# Configuración variables de entorno

In [ ]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.messages import HumanMessage
from qdrant_client import QdrantClient
from google.colab import userdata
import os

# Variables para APIs
try:
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
    openai_api_key = userdata.get('OPENAI_API_KEY')

    # Configurar Qdrant API Key y URL en variables de entorno para agent_graph.py
    qdrant_api_key = userdata.get('QDRANT_API_KEY')
    qdrant_url = "https://1b3b98fb-f02f-493e-9e7d-69481d5858e5.sa-east-1-0.aws.cloud.qdrant.io:6333"
    collection_name = 'manuales_tecnicos'

    os.environ['QDRANT_API_KEY'] = qdrant_api_key
    os.environ['QDRANT_URL'] = qdrant_url
    os.environ['QDRANT_COLLECTION_NAME'] = collection_name

except Exception as e:
    print(f"Error configurando entorno: {e}")

# Modelos de embedding y llm
embedding = OpenAIEmbeddings(model = 'text-embedding-3-small')
llm = ChatOpenAI(model = 'gpt-5-nano', temperature = 0)

# Cliente para uso local en el notebook
qdrant_client = QdrantClient(
    url = qdrant_url,
    api_key = qdrant_api_key,
)

INFO: HTTP Request: GET https://1b3b98fb-f02f-493e-9e7d-69481d5858e5.sa-east-1-0.aws.cloud.qdrant.io:6333 "HTTP/1.1 200 OK"


# Dataset de validación

In [ ]:
test_dataset = [
    # Lavadora Secadora 11Kg / 7Kg con Eco Bubble™
    {
        'question': "Cuáles son los componentes de la lavadora EcoBubble?",
        'ground_truth': """Son: palanca de seguridad, cajón de detergente, panel de control, puerta, tambor, filtro de pelusa, tubo de desagüe de emergencia, cubierta del filtro, tapa, enchufe, manguera de drenaje, y patas niveladoras.""",
        'product_name': 'Lavadora Secadora EcoBubble',
        'doc_type': 'Manual de usuario'
    },
    {
        'question': "Para el panel de control de la lavadora EcoBubble, cuáles son las opciones de centrifugado y su descripción?",
        'ground_truth': """Presione para cambiar la velocidad de centrifugado para el ciclo seleccionado.
        •	Mantener en remojo (Sin indicador): Se suspende el ciclo de aclarado final para mantener la colada en el agua. Para sacar la ropa, ejecute un ciclo de desagüe o de centrifugado.
        •	Sin Centrifugar  : El tambor no centrifuga después del proceso de drenaje final.
        •	Solo centrifugado : Para hacer funcionar el ciclo de Solo centrifugado, presione Centrifugar durante 3 segundos. Cuando aparece el tiempo del ciclo y la velocidad de centrifugado, presione Centrifugar repetidamente hasta que se seleccione la velocidad deseada de centrifugado. Luego, presione Inicio/Pausa para iniciar el ciclo. El tiempo de centrifugado depende del ciclo seleccionado.""",
        'product_name': 'Lavadora Secadora EcoBubble',
        'doc_type': 'Manual de usuario'
    },
    {
        'question': "Para la lavadora EcoBubble, cuáles son los criterios de clasificación para lavar la ropa?",
        'ground_truth': """Los criterios son los siguientes:
        •	Etiqueta de indicaciones: Clasifique la ropa en algodón, fibras mixtas, sintética, seda, lana y rayón.
        •	Color: Separe la ropa blanca de la ropa de color.
        •	Tamaño: Mezclar prendas de diferente tamaño en el tambor mejora el desempeño del lavado.
        •	Sensibilidad: Lave las prendas delicadas por separado, utilizando una opción de Fácil de planchar para las prendas nuevas de pura lana, cortinas y prendas de seda. Consulte las etiquetas de las prendas""",
        'product_name': 'Lavadora Secadora EcoBubble',
        'doc_type': 'Manual de usuario'
    },
    {
        'question': "Para la lavadora EcoBubble, qué significa intensivo y prelavado con burbujas?",
        'ground_truth': """
        • Intensivo: Para prendas muy sucias. El tiempo de funcionamiento de cada ciclo es superior al normal.
        • Prelavando con burbujas: El Prelavando con burbujas ayuda a eliminar una gran variedad de manchas resistentes. Si selecciona Prelavando con burbujas, la ropa sucia se remoja completamente en burbujas de agua para un lavado eficiente. Prelavando con burbujas está disponible con 5 ciclos, a los que añade 30 minutos: ALGODÓN, SINTÉTICO, ESTERILIZAR CON VAPOR, LAVADO DIARIO y LAVADO+SECADO.""",
        'product_name': 'Lavadora Secadora EcoBubble',
        'doc_type': 'Manual de usuario'
    },
    {
        'question': "Para la lavadora EcoBubble, cuál es la presión del agua adecuada?",
        'ground_truth': """La presión de agua adecuada para esta lavadora es entre 50 kPa y 800 kPa.
        Una presión de agua inferior a 50 kPa puede ocasionar que la válvula de agua no se cierre completamente. O, puede llevar más tiempo llenar el tambor, causando el apagado de la lavadora.
        Los grifos de agua deben estar a no más de 120 cm de la parte trasera de la lavadora para que las mangueras de entrada provistas lleguen a la lavadora.""",
        'product_name': 'Lavadora Secadora EcoBubble',
        'doc_type': 'Manual de usuario'
    },
    {
        'question': "Para la lavadora EcoBubble, cómo se hace el mantenimiento del desagüe de emergencia?",
        'ground_truth': """
        En caso de interrupción del suministro eléctrico, vacíe el agua del tambor antes de sacar la ropa.
        1. Apague y desconecte la lavadora de la toma eléctrica.
        2. Presione con suavidad el área superior de la cubierta del filtro para abrirla.
        3. Coloque un recipiente vacío, espacioso sobre la cubierta y estire el tubo de drenaje de emergencia hacia el recipiente mientras sostiene la tapa del tubo.
        4. Abra la tapa del tubo y deje que el agua en el tubo de drenaje de Emergencia fluya hacia el recipiente.
        5. Cuando termine, cierre la tapa del tubo y vuelva a insertar el tubo. Luego, cierre la cubierta del filtro.
        """,
        'product_name': 'Lavadora Secadora EcoBubble',
        'doc_type': 'Manual de usuario'
    },
    {
        'question': "Para la lavadora EcoBubble, qué pasa si hay espuma en exceso?",
        'ground_truth': """
        •	Asegúrese de usar los tipos de detergente recomendados según corresponda.
        •	Use un detergente de alta eficacia (HE) para evitar la formación de espuma en exceso.
        •	Reduzca la cantidad de detergente para el agua blanda, las cargas pequeñas o las cargas ligeramente sucias.
        •	NO se recomienda un detergente de baja eficacia.
        """,
        'product_name': 'Lavadora Secadora EcoBubble',
        'doc_type': 'Manual de usuario'
    },
    {
        'question': "Puedo instalar la lavadora cerca de un electrodoméstico o material inflamable?",
        'ground_truth': """
        No instale este electrodoméstico cerca de un calefactor o materiales inflamables.
        """,
        'product_name': 'Lavadora Secadora EcoBubble',
        'doc_type': 'Manual de usuario'
    },
    {
        'question': "Cómo conecto la manguera de desague?",
        'ground_truth': """
         La manguera de desagüe puede colocarse en tres posiciones:

         1. Sobre el borde de un lavamanos:
        La manguera de desagüe debe colocarsea una altura de entre 60 cm y 90 cm (*) del suelo.
        Para mantener doblado el caño de la manguera de desagüe, utilice la guía de plástico para la manguera que se suministra. Fije la guía a la pared utilizando un gancho para garantizar un desagüe estable.

        2. En una tubería de desagüe
        La tubería de desagüe debe estar a una altura entre 60 cm y 90 cm. Es aconsejable utilizar un tubo vertical de 65 cm de altura.
        • Para garantizar que la manguera de desagüe permanezca en posición, usar la guía de plástico para mangueras provista.
        • Para prevenir que se forme un sifón con el flujo del agua durante el desagüe, asegurarse de que la manguera de desagüe se introduzca en el tubo de desagüe al menos 15 cm.
        • Para prevenir que la manguera de desagüe se esté moviendo, asegurar la guía de la manguera a la pared.

        Requisitos de la tubería vertical dedesagüe:
        • un diámetro de 5 cm como mínimo
        • capacidad de carga de 60 litros por minuto como mínimo.

        3. En un brazo de la tubería de desagüe del fregadero
        El brazo de la tubería de desagüe debe estar situado por encima del sifón del fregadero a fin de que el extremo de la manguera quede al menos a 60 cm del suelo.

        PRECAUCIÓN
        Retire la tapa del brazo de la tubería de desagüe antes de conectar la tubería.
        """,
        'product_name': 'Lavadora Secadora EcoBubble',
        'doc_type': 'Manual de usuario'
    },
    {
        'question': "Aparece el código de error 5C en la lavadora, qué debo hacer?",
        'ground_truth': """
        No hay desagote de agua.
        • Asegúrese de que la manguera de drenaje no esté congelada ni obstruida.
        • Asegúrese de que la manguera de drenaje esté posicionada correctamente, dependiendo del tipo de conexión.
        • Limpie el filtro de basura ya que puede estar obstruido.
        • Asegúrese de que la manguera de drenaje esté derecha todo el camino al sistema de desagote.
        • Si permanece el código de información, contacte con un centro de atención al cliente.
        """,
        'product_name': 'Lavadora Secadora EcoBubble',
        'doc_type': 'Manual de usuario'
    },
    {
        'question': "Se puede botar la lavadora? qué se recomienda para esto?",
        'ground_truth': """
        Sí se puede desechar la lavadora.

        Este electrodoméstico se fabrica con materiales reciclables. Si decide desechar este
        electrodoméstico, siga la normativa local relacionada con la eliminación de desechos. Corte
        el cable de alimentación para que el electrodoméstico no pueda conectarse a una fuente de
        alimentación. Quite la puerta para que los animales y los niños pequeños no puedan quedar
        atrapados dentro del electrodoméstico.
        """,
        'product_name': 'Lavadora Secadora EcoBubble',
        'doc_type': 'Manual de usuario'
    },
    {
        'question': "Qué se debe hacer si se inunda la lavadora?",
        'ground_truth': """
        Si se inunda el aparato, cierre el agua y la alimentación eléctrica inmediatamente y póngase en contacto con el centro de servicio más cercano.
        •	No toque el enchufe con las manos mojadas.
        •	Se puede provocar una descarga eléctrica.
        """,
        'product_name': 'Lavadora Secadora EcoBubble',
        'doc_type': 'Manual de usuario'
    },
    # 55" OLED S95F 4K Vision AI Smart TV (2025)
    {
        'question': "Para la pantalla oled de samsung, qué pasa si la pantalla parpadea o se pone oscura?",
        'ground_truth': """
        Si su TV parpadea o se atenúa esporádicamente, quizá deba desactivar algunas de las características de eficiencia energética.
        Desactive Optimización del brillo, Solución de ahorro de energía, Iluminación por movimiento o Mejorador de contraste.
        • botón de casa > botón direccional hacia la izquierda > Configuración > Todos los ajustes > General y privacidad > Ahorro de energía > Optimización del brillo Int. Ahora
        • botón de casa > botón direccional hacia la izquierda > Configuración > Todos los ajustes > General y privacidad > Ahorro de energía > Solución de ahorro de energía Int. Ahora
        • botón de casa > botón direccional hacia la izquierda > Configuración > Todos los ajustes > General y privacidad > Ahorro de energía > Iluminación por movimiento Int. Ahora
        • botón de casa > botón direccional hacia la izquierda > Configuración > Todos los ajustes > Imagen > Configuración experta. Mejorador de contraste Int. Ahora

        Algunos modelos OLED reducen automáticamente el brillo de la pantalla cuando la temperatura ambiente es elevada para evitar retención de imágenes.
        Ejecute Prueba de imagen. Cuando la calidad de la imagen demuestra ser normal, verifique la señal del dispositivo
        conectado.

        • botón de casa > botón direccional hacia la izquierda > Configuración > Soporte técnico > Cuidado del dispositivo > Autodiagnóstico > Prueba de imagen Int. Ahora
        """,
        'product_name': 'TV OLED S95F 4K',
        'doc_type': 'Manual de usuario'
    },
    {
        'question': "Para la pantalla oled de samsung, cómo se puede conectar el computador pc por la pantalla de la televisión?",
        'ground_truth': """
        Para conectar el TV a la PC en forma inalámbrica, lea las instrucciones en PC Guía de conexión e intente reconectarlo.
        • botón de casa > botón direccional hacia la izquierda > Dispositivos conectados > Guía de conexión > PC > Compartir pantalla (Inalámbrico) Int. Ahora

        Confirme que el TV y la PC estén conectados a la misma red.
        Para conectar el TV a su dispositivo móvil de forma inalámbrica, lea las instrucciones en: Smartphone > Compartir pantalla (Smart view) en Guía de conexión e intente reconectarlo

        Si el TV tiene dificultades para conectarse a su PC o dispositivo móvil por interferencias de radio del entorno, cambie la frecuencia de la banda de acceso inalámbrica e intente reconectarlo.
        """,
        'product_name': 'TV OLED S95F 4K',
        'doc_type': 'Manual de usuario'
    },
    {
        'question': "En la pantalla oled de samsung, cómo se puede usar internet en ella?",
        'ground_truth': """
        Navegue por Internet en su TV.
        • botón de casa > botón direccional hacia la izquierda > Inicio > Aplicaciones > Internet

        Al ejecutar Internet, puede ver los sitios web visitados recientemente o las recomendaciones destacadas. Cuando selecciona el sitio web deseado, puede obtener acceso inmediato a este.
        Esta función puede no admitirse dependiendo del modelo o del área geográfica.
        Puede usar la función Internet más fácilmente después de conectar un teclado y un mouse.
        Puede desplazarse por las páginas web con el botón de dirección del Control remoto Samsung Smart o el Control remoto.
        Las páginas web pueden ser diferentes de las que se ven en una computadora.
        Antes de utilizar Internet, consulte "Leer antes de usar la función Internet."
        La aplicación de Internet tiene incorporada una Configuración de Samsung Pass (botón de casa > botón direccional hacia la izquierda > Inicio > Aplicaciones > Internet > Menú Internet > Configuración > Samsung Pass).

        Con Samsung Pass, puede iniciar sesión en el sitio web de manera sencilla y segura. Al visitar el sitio web nuevamente,
        podrá iniciar sesión con la autenticación biométrica Samsung Pass en su dispositivo móvil sin ingresar su ID y contraseña.
        No obstante, es posible que este inicio de sesión con Samsung Pass no funcione según la política del sitio web. Es por ello
        que debe haber iniciado sesión en su dispositivo móvil con una cuenta Samsung registrada en Samsung Pass.
        """,
        'product_name': 'TV OLED S95F 4K',
        'doc_type': 'Manual de usuario'
    },
    {
        'question': "Para la pantalla oled de samsung, cómo se puede activar el modo ai y para qué sirve?",
        'ground_truth': """
        Permita que el TV analice el entorno y el contenido que está mirando para que pueda brindarle una mejor experiencia de visualización.
        • botón de casa > botón direccional hacia la izquierda > Configuración > Todos los ajustes > Características avanzadas > Configuración de Modo AI > Modo AI int. Ahora

        En el Modo AI, el TV reconoce y analiza el entorno, el ruido, el contenido y sus patrones de uso para proporcionarle la mejor experiencia de visualización. Puede activar o desactivar las siguientes opciones.
        Crear su entorno de visualización preferido. Esta función puede no admitirse dependiendo del modelo o del área geográfica.
        """,
        'product_name': 'TV OLED S95F 4K',
        'doc_type': 'Manual de usuario'
    },
    {
        'question': "En la pantalla oled de samsung, cómo hago la configuración parental?",
        'ground_truth': """
        Configure los ajustes de seguridad del contenido o la aplicación
         • botón de casa > botón direccional hacia la izquierda > Configuración > Todos los ajustes > General y privacidad > Configuración parental Int. Ahora

         Limite el acceso al contenido o las aplicaciones que requieren control de los padres. Solo se puede acceder al contenido o las aplicaciones bloqueados ingresando la contraseña.
         • Aplicar el bloqueo de canales Int. Ahora
         Al seleccionar el menú puede activar o desactivar la función Aplicar el bloqueo de canales.
         Bloquee los canales específicos para evitar que los niños miren contenido para adultos.
         Para usar esta función, se requiere el número PIN.

         • Configuración de Bloqueo de canales Int. Ahora
         Configure el canal en bloqueado o desbloqueado.

         • Bloqueo de aplicación Int. Ahora
         Configure la aplicación instalada para bloquearla o desbloquearla
        """,
        'product_name': 'TV OLED S95F 4K',
        'doc_type': 'Manual de usuario'
    },
    {
        'question': "Para la pantalla oled de samsung, cómo puedo hacer para controlar la tv mediante el celular?",
        'ground_truth': """
        Controle el TV mediante el control remoto para dispositivos móviles.
        Utilice las funciones del control remoto móvil desde la aplicación SmartThings en su dispositivo móvil. Si el TV no está registrado en la aplicación móvil SmartThings, abra el control remoto móvil utilizando el siguiente método.
        • Use el botón de encendido del TV para encenderlo y escanee el código QR que aparece en la pantalla.
        • Antes de usar el control remoto móvil, consulte la guía del usuario provista por el control remoto móvil.
        Es posible que los usuarios tengan que actualizar la aplicación SmartThings a la versión más reciente.
        Es posible que esta función no se admita, dependiendo del modelo.
        Las imágenes, los botones y las funciones del control remoto móvil pueden variar según el modelo o el área geográfica
        """,
        'product_name': 'TV OLED S95F 4K',
        'doc_type': 'Manual de usuario'
    },
    {
        'question': "Para la pantalla samsung, cómo puedo conectar la TV con el mause o el teclado del computador?",
        'ground_truth': """
        Es más fácil controlar el TV mediante la conexión de un teclado, mouse o un controlador para juegos.
        Puede conectar un teclado, mouse o controlador de juegos para controlar con facilidad el TV.

        Conexión de un teclado, mouse o controlador de juegos USB
        • botón de casa > botón direccional hacia la izquierda > Dispositivos conectados > Guía de conexión > Dispositivo de entrada

        Conecte el cable del teclado, mouse o controlador de juegos al puerto USB.
        Es posible que esta función no sea compatible con algunas aplicaciones o dispositivos externos.
        Los controladores de juegos XInput USB son compatibles.
        """,
        'product_name': 'TV OLED S95F 4K',
        'doc_type': 'Manual de usuario'
    },
    {
        'question': "Se pueden importar imágenes desde una memoria USB? Si es así, cómo se hace?",
        'ground_truth': """
        Sí, se puede.

        Es posible que esta función no se admita, dependiendo del modelo.
        1. Conecte el dispositivo de memoria USB al TV.
        2. La memoria USB se reconoce automáticamente y la pantalla muestra una lista de archivos de imágenes, música y videos que están almacenados en dicha memoria.
        Una forma alternativa para ejecutar la memoria USB consiste en acceder a:
        • botón de casa > botón direccional hacia la izquierda > Dispositivos conectados y seleccionar la memoria USB.
        3. Acceda a la carpeta que contiene el archivo de imagen que desea guardar en el TV y, luego, seleccione Opción > Enviar a Modo Arte.
        4. Después de seleccionar una imagen, seleccione Enviar para ver las imágenes enviadas en Mis fotos o establecer configuraciones en el modo Arte.

        Resoluciones recomendadas (16:9): 1920 x 1080 (modelo de 32 pulgadas), 3840 x 2160 (modelos de 43 pulgadas o más
        """,
        'product_name': 'TV OLED S95F 4K',
        'doc_type': 'Manual de usuario'
    },
    {
        'question': "Por qué mi tele no se enciende?",
        'ground_truth': """
        Si está teniendo problemas para encender su TV, hay varias cosas que puede verificar antes de llamar al departamento de servicio.
        Confirme que el cable de alimentación del TV esté conectado correctamente en ambos extremos y que el control remoto esté funcionando normalmente.
        Asegúrese de que el cable de la antena o el cable de la TV por cable esté bien conectado.
        Si tiene un decodificador de cable o satélite, confirme que esté enchufado y encendido.
        Si su modelo de televisor es compatible con la caja One Connect, verifique el Cable One Connect al conectar la TV a la caja One Connect o verifique la conexión entre el TV y la caja One Connect.
        """,
        'product_name': 'TV OLED S95F 4K',
        'doc_type': 'Manual de usuario'
    },
    {
        'question': "Cómo puedo cambiar el idioma del menu? Me aparece en otro idioma.",
        'ground_truth': """
        Cambio del idioma del menú
        • botón de casa > botón direccional hacia la izquierda > Configuración > Todos los ajustes > General y privacidad > Idioma Int. Ahora
        """,
        'product_name': 'TV OLED S95F 4K',
        'doc_type': 'Manual de usuario'
    },
    # Refrigerador French Door
    {
        'question': "Para el refrigerador, mencióname las funciones del producto junto con sus descripciones.",
        'ground_truth': """

        ALARMA DE LA PUERTA
        La función de Alarma de la puerta está diseñada para prevenir el mal fun
        cionamiento del refrigerador en caso de que una puerta del refrigerador
        o cajón del congelador permanecieran abiertos. Si se deja abierta una
        puerta del refrigerador o abierto el cajón del congelador por más de 60
        segundos, sonará una alarma de advertencia en intervalos de 30 segun
        dos.

        CAJONES CON CONTROL DE HUMEDAD
        Los cajones con control de humedad están diseñados para ayudar a
        mantener las frutas y las verduras frescas. Para controlar la cantidad de
        humedad en los cajones, ajuste las configuraciones entre Fruit (Fruta) y
        Vegetable (Vegetales).

        GLIDE‘N’SERVE
        Glide‘N’Serve proporciona un espacio de almacenamiento con control
        de temperatura variable que mantiene el compartimento más frío que
        el resto del refrigerador. Es un lugar conveniente para almacenar sánd
        wiches o carnes a cocinar.

        BISAGRA CON MECANISMO DE CIERRE AUTOMÁTICO
        Las puertas del refrigerador y los cajones del congelador se cierran au
        tomáticamente al empujarlos suavemente. (La puerta sólo se cierra au
        tomáticamente cuando está abierta a un ángulo menor a 30°).

        ICE PLUS
        La producción de hielo puede aumentarse en aproximadamente un 20
        porciento cuando la sección del congelador se mantiene a la temperatura
        más fría durante un período de 24 horas.
        """,
        'product_name': 'Refrigerador French Door',
        'doc_type': 'Manual de usuario'
    },
    {
        'question': "Para el refrigerador, dime las partes del interiores del refrigerador.",
        'ground_truth': """Son: Lámparas interiores LED, estante de refrigerador ajustable, bandeja para lácteos, Bandejas de puerta modulares,
        Bisagra con mecanismo de cierre automático, Glide'N'Serve, Durabase y divisor Durabase, Cajón extraíble, Depósito de hielo, Máquina de hielo,
        Cajón, y Bandeja de puerta fija.
        """,
        'product_name': 'Refrigerador French Door',
        'doc_type': 'Manual de usuario'
    },
    {
        'question': "Cuál debe ser la temperatura para que el refrigerador se pueda instalar?",
        'ground_truth': """
        Instale este electrodoméstico en una zona donde la temperatura esté entre los 13°C (55°F) y 43°C (110°F).
        """,
        'product_name': 'Refrigerador French Door',
        'doc_type': 'Manual de usuario'
    },
    {
        'question': "En el refrigerador instalé la máquina de hielos, y el hielo sabe mal y huele mal. A qué se debe? Cómo lo puedo arreglar?",
        'ground_truth': """

        Causa:
        • El suministro de agua contiene minerales como el azufre.
        Solución:
        • Es posible que sea necesario instalar un filtro de agua para eliminar los problemas de sabor y olor. NOTA: En algunos casos, es posible que el filtro no sea de ayuda.
        Puede que no sea posible eliminar todos los minerales/olor/sabor en todos los suministros de agua

        Causa:
        • La máquina de hielo se instaló recientemente.
        Solución:
        • El hielo que lleva demasiado tiempo almacenado se encogerá, nublará, y puede desarrollar mal sabor. Deseche el hielo viejo y haga nuevo.

        Causa:
        • Los alimentos no están bien guardados en los compartimentos.
        Solución:
        • Vuelva a envolver los alimentos. Es posible que los olores se trasladen al hielo si los alimentos no están bien envueltos.

        Causa:
        • Es necesario limpiar el interior del refrigerador.
        Solución:
        • Para más información, consulte la sección de Cuidado y Limpieza.

        Causa:
        • Es necesario limpiar el contenedor en donde se almacena el hielo.
        Solución:
        • Vacíe y lave el contenedor (deseche los cubitos viejos). Asegúrese de que el contenedor esté completamente seco antes de volver a instalarlo.
        """,
        'product_name': 'Refrigerador French Door',
        'doc_type': 'Manual de usuario'
    },
    {
        'question': "Cómo se monta la puerta izquierda del refrigerador?",
        'ground_truth': """
        Instale la puerta izquierda del refrigerador luego de instalar la puerta derecha.
        1. Asegúrese de que el mango plástico esté colocado en la parte inferior de la puerta. Instale la puerta del refrigerador sobre la bisagra central.
        2. Alinee la puerta con el cuerpo del refrigerador.
        3. Alinee los agujeros de la bisagra superior con los agujeros de la parte superior del refrigerador. Inserte y ajuste tres pernos en la bisagra.
        4. Apriete el tornillo de conexión a tierra.
        5. Reconecte el mazo de cables.
        6. Vuelva a colocar la cubierta de la bisagra. Inserte y apriete los tornillos de la cubierta
        """,
        'product_name': 'Refrigerador French Door',
        'doc_type': 'Manual de usuario'
    },
    {
        'question': "A qué presión debe estar el agua para conectar las tuberías de agua con el refrigerador?",
        'ground_truth': """La presión del agua debe estar entre los 20 y los 120 psi (0.14 y 0.82 MPa) en modelos sin filtro de agua.
        Y, entre los 40 y 120 psi (0.28 y 0.82 MPa) en modelos con filtro de agua.
        """,
        'product_name': 'Refrigerador French Door',
        'doc_type': 'Manual de usuario'
    },
    {
        'question': "Por qué las puertas del refrigerador son difíciles de abrir?",
        'ground_truth': """
        Causa:
        • Las juntas están sucias o pegajosas.
        Solución:
        • Limpie las juntas y las superficies de contacto. Aplique una fina capa de abrillantador para electrodomésticos o cera de cocina en las juntas después de limpiarlas.

        Causa:
        • La puerta se cerró recientemente
        Solución:
        • Cuando abre la puerta, ingresa aire más caliente al refrigerador. Al enfriarse ese aire caliente, puede crear un vacío. Si le cuesta abrir la puerta, espere un minuto
        para permitir que la presión del aire se equilibre, y luego pruebe para ver si se abre con más facilidad.
        """,
        'product_name': 'Refrigerador French Door',
        'doc_type': 'Manual de usuario'
    },
    {
        'question': "A qué distancia de la muralla debo instalar el refrigerador?",
        'ground_truth': """
        Deje al menos 61 cm (24 pulgadas) en el frente del refrigerador para abrir las puertas y al menos 5,08 cm (2 pulgadas) entre la parte posterior del refrigerador y la pared.
        """,
        'product_name': 'Refrigerador French Door',
        'doc_type': 'Manual de usuario'
    },
    {
        'question': "Cuándo suena la alarma de la puerta del refrigerador?",
        'ground_truth': """Si se deja abierta una puerta del refrigerador o abierto el cajón del congelador por más de 60 segundos, sonará una alarma de advertencia en intervalos de 30 segundos.""",
        'product_name': 'Refrigerador French Door',
        'doc_type': 'Manual de usuario'
    },
    # ROG Xbox Ally X (2025) RC73XA Consola Portátil
    {
        'question': "A qué temperatura debe operar la consola?",
        'ground_truth': """Este producto solo debe usarse en entornos con una temperatura ambiente de entre 5°C (41°F) y 35°C (95°C).""",
        'product_name': 'ROG Xbox Ally X',
        'doc_type': 'Manual de usuario'
    },
    {
        'question': "Cómo puedo hacer para que la batería de la consola dure más?",
        'ground_truth': """
        Cuidado de la batería estándar
        • Si no va a utilizar el dispositivo durante un período de tiempo prolongado, asegúrese de cargar la batería al 50%, y, a continuación,
        apague el dispositivo y desconecte el adaptador de alimentación de CA. Recargue la batería al 50% cada tres meses para evitar la sobredescarga y los daños en la batería.
        • Para prolongar la vida útil de la batería, evite cargarla con un alto voltaje durante un período de tiempo prolongado. Si usa constantemente alimentación de CA para el dispositivo, asegúrese de descargar la batería al 50% al menos una vez cada dos semanas.
        Para ayudar a prolongar la vida útil de la batería, también puede ajustar la configuración de Battery Health Charging (Estado de carga de la batería) en MyASUS.
        • Se recomienda almacenar la batería a temperaturas entre 5°C (41°F) y 35°C (95°F) con la carga de la batería al 50%. Para ayudar a prolongar la vida útil de la batería, también puede ajustar la configuración de Battery Health Charging (Estado de carga de la batería) en MyASUS.
        • No deje la batería en ambientes húmedos. La exposición a entornos húmedos puede aumentar la tasa de sobredescarga de la batería. Las bajas temperaturas pueden dañar los productos químicos de la batería, y las altas o el sobrecalentamiento provocar un riesgo de explosión.
        • No coloque el dispositivo ni la batería cerca de radiadores, chimeneas, hornos, calentadores o cualquier fuente de calor con una temperatura superior a 60°C (140°F). Las altas temperaturas próximas podrían provocar una explosión o fuga que podría desembocar en un incendio.
        """,
        'product_name': 'ROG Xbox Ally X',
        'doc_type': 'Manual de usuario'
    },
    {
        'question': "Cómo se puede acceder a la BIOS de la consola?",
        'ground_truth': """
        Para introducir la configuración del BIOS, utilice cualquiera de los siguientes métodos:

        • Reinicie la consola portátil ROG y mantenga pulsado el botón para bajar el volumen durante la POST.
        • Inicie el menú START (Inicio) y seleccione Settings (Configuración) > System (Sistema) > Recovery (Recuperación), y a continuación, seleccione Restart now (Reiniciar ahora) en Advanced startup (Inicio avanzado). Cuando aparezca la pantalla Advanced startup (Inicio avanzado), seleccione Troubleshoot (Solución de problemas) > Advanced Options (Opciones avanzadas) > UEFI firmware Settings (Configuración de firmware UEFI) > Restart (Reiniciar)
        """,
        'product_name': 'ROG Xbox Ally X',
        'doc_type': 'Manual de usuario'
    },
    {
        'question': "Cómo puedo conectar la consola con bluetooth",
        'ground_truth': """
        Utilice Bluetooth para facilitar las transferencias de datos inalámbricas con otros dispositivos habilitados para Bluetooth.

        ¡IMPORTANTE! Airplane mode (Modo avión) deshabilita esta función. Asegúrese de desactivar Airplane Mode (Modo avión) antes de habilitar la conexión Bluetooth de su consola portátil ROG.

        Emparejamiento con otros dispositivos habilitados para Bluetooth
        Para poder hacer transferencias de datos, deberá emparejar su consola portátil ROG con otros dispositivos habilitados para Bluetooth. Para conectar sus dispositivos, siga estos pasos:

        1. Pulse el botón de Xbox para abrir Game Bar > Settings (Configuración) > Bluetooth.
        2. Seleccione Add device (Añadir dispositivo) para buscar dispositivos con Bluetooth.
        3. Seleccione un dispositivo de la lista para emparejar su consola portátil ROG con el dispositivo.

        NOTA: En algunos dispositivos habilitados para Bluetooth, puede que se le solicite introducir la contraseña de su consola portátil ROG.
        """,
        'product_name': 'ROG Xbox Ally X',
        'doc_type': 'Manual de usuario'
    },
    {
        'question': "Cómo conecto la consola al wifi?",
        'ground_truth': """
        Para conectar su consola portátil ROG a la red wifi, siga estos pasos:

        1. Pulse el botón de Xbox para abrir Game Bar > Settings (Configuración) > Network (Red).
        2. Seleccione un punto de acceso de la lista de conexiones Wi-Fi disponibles.
        3. Seleccione Connect (Conectar) para iniciar la conexión de red.

        NOTA: Puede que se le solicite una clave de seguridad para activar la conexión Wi-Fi.
        """,
        'product_name': 'ROG Xbox Ally X',
        'doc_type': 'Manual de usuario'
    },
    {
        'question': "Puedo retirar la batería de la consola?",
        'ground_truth': """
        ¡ADVERTENCIA!
        • Solo los técnicos autorizados por ASUS deben extraer la batería del interior del dispositivo (solo para baterías no extraíbles).
        • La extracción o el desmontaje de la batería de este dispositivo puede implicar un riesgo de incendio o quemaduras químicas.
        • Respete las etiquetas de advertencia por su propia seguridad.
        • Si se reemplaza la batería por una de un tipo incorrecto, existe riesgo de explosión.
        • No arroje la batería al fuego.
        • No intente cortocircuitar la batería del dispositivo.
        • No intente desmontar y volver a montar la batería (solo para baterías no extraíbles).
        • Si detecta alguna fuga, deje de usar la batería.
        • La batería y sus componentes deben ser reciclados o eliminados correctamente.
        • Mantenga la batería y otros componentes pequeños fuera del alcance de los niños.
        """,
        'product_name': 'ROG Xbox Ally X',
        'doc_type': 'Manual de usuario'
    },
    {
        'question': "Puedo tirar la batería del dispositovo a la basura de mi hogar?",
        'ground_truth': """No arroje la batería a la basura doméstica.
        Utilice una esponja de celulosa limpia o una gamuza de celulosa humedecida con agua tibia.
        """,
        'product_name': 'ROG Xbox Ally X',
        'doc_type': 'Manual de usuario'
    },
    {
        'question': "Qué materiales se deben usar para limpiar el dispositivo?",
        'ground_truth': """
        Utilice una esponja de celulosa limpia o una gamuza de celulosa humedecida con agua tibia. """,
        'product_name': 'ROG Xbox Ally X',
        'doc_type': 'Manual de usuario'
    },
    # Garantías
    {
        'question': "La garantía de la consola asus, hasta cuanto tiempo es válida?",
        'ground_truth': """La garantía es de 12 meses a partir de la fecha de compra.""",
        'product_name': 'Garantía para consola ROG Xbox Ally X',
        'doc_type': 'Política de Garantía'
    },
    {
        'question': "La garantía de la pantalla oled, hasta cuántos meses es válida?",
        'ground_truth': """La garantía legal del Producto será de 6 meses, y comienza con la fecha original de compra, indicada en la boleta o factura o cualquier otro documento que acredite la compra o la fecha de recepción del producto, indicada en la guía de despacho o documento similar.""",
        'product_name': 'Garantía multi-producto',
        'doc_type': 'Política de Garantía'
    },
        {
        'question': "En qué consiste la garantía voluntaria que ofrece samsung?",
        'ground_truth': """La garantía voluntaria que ofrece Samsung Electronics Chile Ltda, consistirá exclusivamente en la reparación del Producto""",
        'product_name': 'Garantía multi-producto',
        'doc_type': 'Política de Garantía'
    }
]

# Validación

In [ ]:
import pandas as pd
import time
from langchain_core.messages import HumanMessage, ToolMessage
# `get_openai_callback` Cuenta todas las llamadas a la API de OpenAI en su conjunto, es decir, el total
from langchain_community.callbacks import get_openai_callback
from agent_graph import graph

def extract_retrieved_contexts(messages):
    """
    Extrae el texto recuperado de los ToolMessage en el historial del agente.
    """
    contexts = []
    for message in messages:
        if isinstance(message, ToolMessage):
            content = message.content
            chunks = content.split("\n\n------\n\n")
            contexts.extend(chunks)
    return contexts

results = {
    'question': [],
    'answer': [],
    'contexts': [],
    'ground_truth': [],
    'start_time': [],
    'end_time': [],
    'input_tokens': [],
    'output_tokens': [],
    'total_tokens': [],
    'total_cost_usd': [],
    'num_calls': []
}
for i, item in enumerate(test_dataset):
    # Simular input como lo hace app.py
    context_system = f"""
    [META-DATA DEL SISTEMA]
    Producto seleccionado: {item['product_name']}
    Tipo de consulta: {item.get('doc_type', 'general')}
    """
    user_input = f"{context_system}\nConsulta del usuario: {item['question']}"
    inputs = {
        'messages': [HumanMessage(content = user_input)],
        'input_type': 'text',
        'num_retries': 0
    }
    start_ts = time.time()
    # Ejecutar el grafo
    try:
        config = {'configurable': {'thread_id': f"eval_{i}"}}

        # Envolver la ejecución con el callback
        # Acá funciona como un contador automático, que por ejemplo, suma los tokens por llamada hasta tener la cuenta final (end-to-end)
        with get_openai_callback() as cb:
            output_state = graph.invoke(inputs, config = config)
            # Aquí 'cb' ya tiene toda la información acumulada de esta ejecución
            tokens_in = cb.prompt_tokens
            tokens_out = cb.completion_tokens
            tokens_total = cb.total_tokens
            cost_usd = cb.total_cost
            num_calls = cb.successful_requests

        end_ts = time.time()

        final_answer = output_state['final_response']
        retrieved_contexts = extract_retrieved_contexts(output_state['messages'])

        results['question'].append(item['question'])
        results['answer'].append(final_answer)
        results['contexts'].append(retrieved_contexts)
        results['ground_truth'].append(item['ground_truth'])
        results['start_time'].append(start_ts)
        results['end_time'].append(end_ts)
        results['input_tokens'].append(tokens_in)
        results['output_tokens'].append(tokens_out)
        results['total_tokens'].append(tokens_total)
        results['total_cost_usd'].append(cost_usd)
        results['num_calls'].append(num_calls)

    except Exception as e:
        end_ts = time.time()
        print(f"❌ Error en caso {i}: {e}")
        # Guardar valores vacíos para no romper el array
        results['question'].append(item['question'])
        results['answer'].append('Error')
        results['contexts'].append(['Error'])
        results['ground_truth'].append(item['ground_truth'])
        results['start_time'].append(start_ts)
        results['end_time'].append(end_ts)
        results['input_tokens'].append(0)
        results['output_tokens'].append(0)
        results['total_tokens'].append(0)
        results['total_cost_usd'].append(0)
        results['num_calls'].append(0)

INFO: NumExpr defaulting to 2 threads.
INFO: HTTP Request: GET https://1b3b98fb-f02f-493e-9e7d-69481d5858e5.sa-east-1-0.aws.cloud.qdrant.io:6333 "HTTP/1.1 200 OK"
INFO: HTTP Request: PUT https://1b3b98fb-f02f-493e-9e7d-69481d5858e5.sa-east-1-0.aws.cloud.qdrant.io:6333/collections/manuales_tecnicos/index?wait=true "HTTP/1.1 200 OK"
INFO: HTTP Request: PUT https://1b3b98fb-f02f-493e-9e7d-69481d5858e5.sa-east-1-0.aws.cloud.qdrant.io:6333/collections/manuales_tecnicos/index?wait=true "HTTP/1.1 200 OK"
INFO: >>> [ROUTER] Input detectado: text
INFO: >>> [NODO] text_node: Nodo usado
INFO: >>> [NODO] agent_node: Iniciando nodo.
INFO: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO: >>> [NODO] tool_call_node: Ejecutando herramienta.
INFO:     Procesando llamada a: {'name': 'consultar_manuales', 'args': {'query': '¿Cuáles son los componentes de la lavadora EcoBubble?', 'product_name': 'Lavadora Secadora EcoBubble', 'doc_type': 'Manual de usuario'}, 'id': 'cal

# Cálculo de métricas mediante RAGAS

In [ ]:
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
)
# Crear Dataset de HuggingFace (Formato que pide RAGAS)
ragas_dataset = Dataset.from_dict(results)
# Se usa un modelo potente para evaluar (Juez)
llm_evaluator = ChatOpenAI(model = 'gpt-5-mini')

for metric in [faithfulness, answer_relevancy, context_precision]:
    # Dependiendo de la versión de RAGAS, a veces es necesario inyectar el LLM así
    if hasattr(metric, 'llm'):
        metric.llm = llm_evaluator

# Configurar Métricas y Modelos
metrics = [
    faithfulness,
    answer_relevancy,
    context_precision,
]

# jecutar Evaluación
evaluation_results = evaluate(
    dataset = ragas_dataset,
    metrics = metrics,
    llm = llm_evaluator,
    embeddings = embedding
)

/tmp/ipython-input-5126984.py:3: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import (
/tmp/ipython-input-5126984.py:3: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import (
/tmp/ipython-input-5126984.py:3: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_precision
  from ragas.metrics import (


Evaluating:   0%|          | 0/126 [00:00<?, ?it/s]

INFO: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.openai.com/v1/chat/completion

# Exportar datos a hoja de cálculo

In [ ]:
import polars as pl

print(f"Resultados Globales: {evaluation_results}")
df_1 = pl.DataFrame(evaluation_results.to_pandas())

df_2 = pl.DataFrame(results) \
    .with_columns(
        (pl.col('end_time') - pl.col('start_time')).round(2).alias('latency')
    )

df_results = df_1 \
    .join(df_2, left_on = 'user_input', right_on = 'question', how = 'inner') \
    .select(
        pl.col('user_input'),
        pl.col('retrieved_contexts'),
        pl.col('response'),
        pl.col('reference'),
        pl.col('faithfulness'),
        pl.col('answer_relevancy'),
        pl.col('context_precision'),
        pl.col('start_time'),
        pl.col('end_time'),
        pl.col('latency'),
        pl.col('input_tokens'),
        pl.col('output_tokens'),
        pl.col('total_tokens'),
        pl.col('total_cost_usd'),
        pl.col('num_calls')
    )

file_name = 'resultados_evaluacion.xlsx'
if not os.path.exists(file_name):
    df_results.write_excel(file_name)
    print(f"\n📁 Archivo {file_name} guardado.")
else:
    print(f"\nYa existe archivo: {file_name}")

Resultados Globales: {'faithfulness': 0.9163, 'answer_relevancy': 0.6635, 'context_precision': 0.7862}

📁 Archivo resultados_evaluacion.xlsx guardado.
